# Wholebody Pose Estimation using SAPIENS Model

This notebook predicts wholebody poses (body + hands + face) using the SAPIENS model.
It processes videos from the inputs/{data_collection} directory and outputs wholebody keypoints.

In [493]:
# First, let's try to fix the numpy incompatibility issue

import os
import cv2
import numpy as np
import mmcv
import json
import torch
from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image

# Import the necessary MMPose components
import mmengine
from mmpose.apis import inference_topdown
from mmpose.apis import init_model as init_pose_estimator
from mmpose.structures import merge_data_samples, split_instances
from mmpose.registry import VISUALIZERS

# IMPORTANT: Import mmpretrain to register the ViT model
# import mmpretrain
# import mmpretrain.models

# Import custom helpers
from helpers.definitions import *

## Configuration

Set up the data collection path and other configuration parameters.

In [494]:
# Configuration
data_collection = 'cha/synced_cut'  # Change this to match your data collection
show = True  # Show visualizations
draw_bbox = True  # Draw bounding boxes
kpt_thresh = 0.3  # Keypoint confidence threshold

labeled_frames = list(range(0, 300, 30))

# Create output directories
pose_output_dir = 'predictions_2d/'
vis_output_dir = 'vis_dir/'
methods = ['rtmpose', 'dwpose']

for method in methods:
    os.makedirs(os.path.join(pose_output_dir, method, data_collection), exist_ok=True)
    os.makedirs(os.path.join(vis_output_dir, method, data_collection), exist_ok=True)

# Input data directory
data_dir = f'inputs/{data_collection}'

# Ensure data directory exists
if not os.path.exists(data_dir):
    raise FileNotFoundError(f"Data directory not found: {data_dir}")

## Initialize SAPIENS Model

Initialize the SAPIENS model directly without using the helper function.

In [495]:
if 'sapiens' in methods:
    # Path to SAPIENS model configuration and weights
    SAPIENS_CONFIG = 'configs/wholebody_2d_keypoint/sapiens/sapiens_1b-210e_coco_wholebody-1024x768.py'
    SAPIENS_WEIGHTS = 'checkpoints/sapiens/sapiens_1b_coco_wholebody_best_coco_wholebody_AP_727.pth'

    # Set visualization parameters
    POSE_VIS_RADIUS = 3
    POSE_VIS_ALPHA = 0.8
    POSE_VIS_LINE_WIDTH = 2
    POSE_KPT_THRESHOLD = 0.3

    # Initialize SAPIENS model
    try:
        print("Initializing SAPIENS model...")
        pose_estimator_sapiens = init_pose_estimator(
            SAPIENS_CONFIG,
            SAPIENS_WEIGHTS,
            device='cuda',
            cfg_options=dict(model=dict(test_cfg=dict(output_heatmaps=False)))
        )
        
        # Configure visualizer
        pose_estimator_sapiens.cfg.visualizer.radius = POSE_VIS_RADIUS
        pose_estimator_sapiens.cfg.visualizer.alpha = POSE_VIS_ALPHA
        pose_estimator_sapiens.cfg.visualizer.line_width = POSE_VIS_LINE_WIDTH
        visualizer_sapiens = VISUALIZERS.build(pose_estimator_sapiens.cfg.visualizer)
        visualizer_sapiens.set_dataset_meta(pose_estimator_sapiens.dataset_meta, skeleton_style='mmpose')
        
        print("SAPIENS model initialized successfully!")
    except Exception as e:
        print(f"Error initializing SAPIENS model: {e}")
        import traceback
        traceback.print_exc()

In [496]:
if 'rtmpose' in methods:
    RTMPOSE_CONFIG = 'configs/wholebody_2d_keypoint/rtmpose/coco-wholebody/rtmpose-l_8xb32-270e_coco-wholebody-384x288.py'
    RTMPOSE_WEIGHTS = 'checkpoints/wholebody/rtmpose-l_simcc-coco-wholebody_pt-aic-coco_270e-384x288-eaeb96c8_20230125.pth'

    # Set visualization parameters
    POSE_VIS_RADIUS = 3
    POSE_VIS_ALPHA = 0.8
    POSE_VIS_LINE_WIDTH = 2
    POSE_KPT_THRESHOLD = 0.3

    # Initialize RTMPOSE model
    try:
        print("Initializing RTMPOSE model...")
        pose_estimator_rtmpose = init_pose_estimator(
            RTMPOSE_CONFIG,
            RTMPOSE_WEIGHTS,
            device='cuda',
            cfg_options=dict(model=dict(test_cfg=dict(output_heatmaps=False)))
        )
        
        # Configure visualizer
        pose_estimator_rtmpose.cfg.visualizer.radius = POSE_VIS_RADIUS
        pose_estimator_rtmpose.cfg.visualizer.alpha = POSE_VIS_ALPHA
        pose_estimator_rtmpose.cfg.visualizer.line_width = POSE_VIS_LINE_WIDTH
        visualizer_rtmpose = VISUALIZERS.build(pose_estimator_rtmpose.cfg.visualizer)
        visualizer_rtmpose.set_dataset_meta(pose_estimator_rtmpose.dataset_meta, skeleton_style='mmpose')

        print("RTMPOSE model initialized successfully!")
    except Exception as e:
        print(f"Error initializing RTMPOSE model: {e}")
        import traceback
        traceback.print_exc()

Initializing RTMPOSE model...
Loads checkpoint by local backend from path: checkpoints/wholebody/rtmpose-l_simcc-coco-wholebody_pt-aic-coco_270e-384x288-eaeb96c8_20230125.pth
RTMPOSE model initialized successfully!


In [497]:
if 'dwpose' in methods:
    DWPOSE_CONFIG = 'configs/wholebody_2d_keypoint/rtmpose/ubody/rtmpose-l_8xb32-270e_coco-ubody-wholebody-384x288.py'
    DWPOSE_WEIGHTS = 'checkpoints/wholebody/rtmpose-l_simcc-ucoco_dw-ucoco_270e-384x288-2438fd99_20230728.pth'

    # Set visualization parameters
    POSE_VIS_RADIUS = 3
    POSE_VIS_ALPHA = 0.8
    POSE_VIS_LINE_WIDTH = 2
    POSE_KPT_THRESHOLD = 0.3

    # Initialize RTMPOSE model
    try:
        print("Initializing DWPOSE model...")
        pose_estimator_dwpose = init_pose_estimator(
            DWPOSE_CONFIG,
            DWPOSE_WEIGHTS,
            device='cuda',
            cfg_options=dict(model=dict(test_cfg=dict(output_heatmaps=False)))
        )
        
        # Configure visualizer
        pose_estimator_dwpose.cfg.visualizer.radius = POSE_VIS_RADIUS
        pose_estimator_dwpose.cfg.visualizer.alpha = POSE_VIS_ALPHA
        pose_estimator_dwpose.cfg.visualizer.line_width = POSE_VIS_LINE_WIDTH
        visualizer_dwpose = VISUALIZERS.build(pose_estimator_dwpose.cfg.visualizer)
        visualizer_dwpose.set_dataset_meta(pose_estimator_dwpose.dataset_meta, skeleton_style='mmpose')

        print("DWPOSE model initialized successfully!")
    except Exception as e:
        print(f"Error initializing DWPOSE model: {e}")
        import traceback
        traceback.print_exc()

Initializing DWPOSE model...
Loads checkpoint by local backend from path: checkpoints/wholebody/rtmpose-l_simcc-ucoco_dw-ucoco_270e-384x288-2438fd99_20230728.pth
DWPOSE model initialized successfully!


## Initialize YOLO Model for Person Detection

We'll use YOLOv8 for person detection before applying the SAPIENS pose estimator.

In [498]:
# Initialize YOLO model for person detection
from ultralytics import YOLO

model_person = YOLO('checkpoints/yolo/yolo11l.pt')
model_person.conf = 0.5  # Confidence threshold

def detect_person(frame):
    """Detect person bounding boxes in a frame using YOLO.
    
    Args:
        frame: Input frame as numpy array
        
    Returns:
        numpy array of person bounding boxes in format [x1, y1, x2, y2]
    """
    results = model_person(frame, classes=0)  # Class 0 is person
    boxes = []
    
    for r in results:
        boxes_tensor = r.boxes.xyxy.cpu()
        confs = r.boxes.conf.cpu()
        
        for box, conf in zip(boxes_tensor, confs):
            if conf > model_person.conf:
                boxes.append(box.numpy())
    
    return np.array(boxes) if boxes else np.array([])

## SAPIENS Pose Estimation Function

Create functions to estimate poses and visualize them.

In [499]:
def estimate_pose(img, bbox, pose_estimator, visualizer, show=False, write_img=None):
    """Estimate wholebody pose.
    
    Args:
        img: Input image (path or numpy array)
        bbox: Bounding box coordinates [[x1,y1,x2,y2]]
        pose_estimator: Initialized pose estimator
        show: Whether to show visualization
        write_img: Optional image to draw visualization on
    
    Returns:
        Tuple of (body_instances, left_hand_instances, right_hand_instances)
    """
    # Ensure bbox is in correct format (N,4)
    if isinstance(bbox, np.ndarray):
        bbox = bbox.reshape(1,-1) if bbox.size == 4 else bbox

    # Predict keypoints using SAPIENS wholebody model
    with torch.inference_mode():
        pose_results = inference_topdown(pose_estimator, img, bbox)
    
    data_samples = merge_data_samples(pose_results)
    
    # Visualize if requested
    if visualizer is not None and show:
        if write_img is None:
            write_img = img
        if isinstance(write_img, str):
            write_img = mmcv.imread(write_img, channel_order='rgb')
        elif isinstance(write_img, np.ndarray):
            write_img = mmcv.bgr2rgb(write_img)

        visualizer.add_datasample(
            'result',
            write_img,
            data_sample=data_samples,
            draw_gt=False,
            draw_heatmap=False,
            draw_bbox=True,
            show_kpt_idx=False,
            skeleton_style='mmpose',
            show=show,
            wait_time=0.1,
            kpt_thr=POSE_KPT_THRESHOLD)
    
    # Get the predicted instances from the data sample
    pred_instances = data_samples.get('pred_instances', None)
    
    if pred_instances is None:
        return None, None, None
    
    # Extract body, left hand, and right hand keypoints
    # Based on the COCO-WholeBody dataset format used by SAPIENS
    # Body keypoints are the first 17 points (0-16)
    # Left hand starts at index 91 and has 21 keypoints (91-111)
    # Right hand starts at index 112 and has 21 keypoints (112-132)
    
    # Create a copy of the instances to avoid modifying the original
    body_instances = pred_instances.clone()
    left_hand_instances = pred_instances.clone()
    right_hand_instances = pred_instances.clone()
    
    # Extract body keypoints (first 17 keypoints)
    if pred_instances.keypoints.shape[1] > 17:
        body_keypoints = pred_instances.keypoints[:, :17, :]
        body_keypoint_scores = pred_instances.keypoint_scores[:, :17]
        body_instances.keypoints = body_keypoints
        body_instances.keypoint_scores = body_keypoint_scores
    else:
        body_instances = None
        
    # Extract left hand keypoints (indices 91-111)
    if pred_instances.keypoints.shape[1] > 111:
        left_hand_keypoints = pred_instances.keypoints[:, 91:112, :]
        left_hand_keypoint_scores = pred_instances.keypoint_scores[:, 91:112]
        left_hand_instances.keypoints = left_hand_keypoints
        left_hand_instances.keypoint_scores = left_hand_keypoint_scores
    else:
        left_hand_instances = None
        
    # Extract right hand keypoints (indices 112-132)
    if pred_instances.keypoints.shape[1] > 132:
        right_hand_keypoints = pred_instances.keypoints[:, 112:133, :]
        right_hand_keypoint_scores = pred_instances.keypoint_scores[:, 112:133]
        right_hand_instances.keypoints = right_hand_keypoints
        right_hand_instances.keypoint_scores = right_hand_keypoint_scores
    else:
        right_hand_instances = None
        
    return body_instances, left_hand_instances, right_hand_instances

## Visualization Function

Create a function to visualize the detected whole-body keypoints.

In [500]:
def visualize_wholebody_pose(frame, body_instances, left_hand_instances, right_hand_instances):
    """Visualize wholebody pose on a frame.
    
    Args:
        frame: Input frame
        body_instances: Body pose instances
        left_hand_instances: Left hand pose instances
        right_hand_instances: Right hand pose instances
        
    Returns:
        Frame with visualized poses
    """
    vis_frame = frame.copy()
    
    # Draw body keypoints
    if body_instances is not None:
        for i in range(len(body_instances.keypoints)):
            keypoints = body_instances.keypoints[i]
            scores = body_instances.keypoint_scores[i]
            
            # Draw body keypoints with confidence > threshold
            for j, (kp, score) in enumerate(zip(keypoints, scores)):
                if score > kpt_thresh:
                    x, y = map(int, kp)
                    cv2.circle(vis_frame, (x, y), 5, (0, 255, 0), -1)
            
            # Draw connections between keypoints (skeleton)
            # COCO skeleton connections
            connections = [
                (0, 1), (1, 2), (2, 3), (3, 4),  # Head to neck to shoulders
                (5, 6),  # Shoulders
                (5, 7), (7, 9),  # Left arm
                (6, 8), (8, 10),  # Right arm
                (5, 11), (6, 12),  # Shoulders to hips
                (11, 12),  # Hips
                (11, 13), (13, 15),  # Left leg
                (12, 14), (14, 16)  # Right leg
            ]
            
            for conn in connections:
                if scores[conn[0]] > kpt_thresh and scores[conn[1]] > kpt_thresh:
                    pt1 = tuple(map(int, keypoints[conn[0]]))
                    pt2 = tuple(map(int, keypoints[conn[1]]))
                    cv2.line(vis_frame, pt1, pt2, (0, 255, 0), 2)
    
    # Draw left hand keypoints
    if left_hand_instances is not None:
        for i in range(len(left_hand_instances.keypoints)):
            keypoints = left_hand_instances.keypoints[i]
            scores = left_hand_instances.keypoint_scores[i]
            
            # Draw keypoints
            for j, (kp, score) in enumerate(zip(keypoints, scores)):
                if score > kpt_thresh:
                    x, y = map(int, kp)
                    cv2.circle(vis_frame, (x, y), 3, (255, 0, 0), -1)
            
            # Hand connections
            finger_connections = [
                # Thumb
                (0, 1), (1, 2), (2, 3), (3, 4),
                # Index finger
                (0, 5), (5, 6), (6, 7), (7, 8),
                # Middle finger
                (0, 9), (9, 10), (10, 11), (11, 12),
                # Ring finger
                (0, 13), (13, 14), (14, 15), (15, 16),
                # Pinky
                (0, 17), (17, 18), (18, 19), (19, 20)
            ]
            
            for conn in finger_connections:
                if scores[conn[0]] > kpt_thresh and scores[conn[1]] > kpt_thresh:
                    pt1 = tuple(map(int, keypoints[conn[0]]))
                    pt2 = tuple(map(int, keypoints[conn[1]]))
                    cv2.line(vis_frame, pt1, pt2, (255, 0, 0), 1)
    
    # Draw right hand keypoints
    if right_hand_instances is not None:
        for i in range(len(right_hand_instances.keypoints)):
            keypoints = right_hand_instances.keypoints[i]
            scores = right_hand_instances.keypoint_scores[i]
            
            # Draw keypoints
            for j, (kp, score) in enumerate(zip(keypoints, scores)):
                if score > kpt_thresh:
                    x, y = map(int, kp)
                    cv2.circle(vis_frame, (x, y), 3, (0, 0, 255), -1)
            
            # Hand connections (same as left hand)
            finger_connections = [
                # Thumb
                (0, 1), (1, 2), (2, 3), (3, 4),
                # Index finger
                (0, 5), (5, 6), (6, 7), (7, 8),
                # Middle finger
                (0, 9), (9, 10), (10, 11), (11, 12),
                # Ring finger
                (0, 13), (13, 14), (14, 15), (15, 16),
                # Pinky
                (0, 17), (17, 18), (18, 19), (19, 20)
            ]
            
            for conn in finger_connections:
                if scores[conn[0]] > kpt_thresh and scores[conn[1]] > kpt_thresh:
                    pt1 = tuple(map(int, keypoints[conn[0]]))
                    pt2 = tuple(map(int, keypoints[conn[1]]))
                    cv2.line(vis_frame, pt1, pt2, (0, 0, 255), 1)
    
    return vis_frame

## Process Videos

Function to process videos and extract wholebody poses.

In [501]:
def process_video(video_path, output_json_path, output_path=None, output_format="frames", stride=1, show=False, method='sapiens'):
    """Process a video file to extract wholebody poses.
    
    Args:
        video_path: Path to input video file
        output_json_path: Path to save output pose data
        output_path: Path to save visualization (video file or directory for frames)
        output_format: Format to save visualizations ("video" or "frames")
        stride: Number of frames to skip between processing
        show: Whether to display frames during processing
        method: Method to use for pose estimation

    Returns:
        dict containing predicted wholebody poses
    """

    # Helper function to convert numpy arrays to lists recursively
    def convert_numpy_to_list(obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {k: convert_numpy_to_list(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert_numpy_to_list(item) for item in obj]
        # Handle numpy scalar types (like float32, int64, etc)
        elif np.isscalar(obj) and isinstance(obj, (np.generic)):
            return obj.item()  # Convert numpy scalar to native Python type
        else:
            return obj
            
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise FileNotFoundError(f"Cannot open video file: {video_path}")
    
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Initialize video writer if output format is video
    video_writer = None
    frames_dir = None
    
    if output_path:
        if output_format.lower() == "video":
            fourcc = cv2.VideoWriter_fourcc(*'mp4v')
            video_writer = cv2.VideoWriter(
                output_path,
                fourcc,
                fps,
                (width, height)
            )
        elif output_format.lower() == "frames":
            # Create directory for frames if it doesn't exist
            frames_dir = output_path
            os.makedirs(frames_dir, exist_ok=True)
    
    # Dictionary to store prediction results
    body_instances_list = []
    left_hand_instances_list = []
    right_hand_instances_list = []
    
    
    # Process frames
    for frame_idx in range(0, total_frames, stride):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        success, frame = cap.read()
        print(f"Processing frame {int(frame_idx/stride)}/{int(total_frames/stride)}")

        if not success:
            break
        
        # Detect persons in the frame
        person_boxes = detect_person(frame)
        
        # Skip if no person detected
        if len(person_boxes) == 0:
            # Create empty instances for this frame
            body_instances_list.append(dict(frame_id=frame_idx, instances=[]))
            left_hand_instances_list.append(dict(frame_id=frame_idx, instances=[]))
            right_hand_instances_list.append(dict(frame_id=frame_idx, instances=[]))
            
            # Write original frame to output if needed
            if video_writer:
                video_writer.write(frame)
            elif frames_dir:
                frame_filename = os.path.join(frames_dir, f"frame_{frame_idx:06d}.png")
                cv2.imwrite(frame_filename, frame)  
            continue

        if method == 'sapiens':
            pose_estimator = pose_estimator_sapiens
            # Estimate wholebody pose
            body_instances, left_hand_instances, right_hand_instances = estimate_pose(
                frame, person_boxes, pose_estimator_sapiens, visualizer_sapiens, show=False
            )
        elif method == 'rtmpose':
            pose_estimator = pose_estimator_rtmpose
            body_instances, left_hand_instances, right_hand_instances = estimate_pose(
                frame, person_boxes, pose_estimator_rtmpose, visualizer_rtmpose, show=False
            )
        elif method == 'dwpose':
            pose_estimator = pose_estimator_dwpose
            body_instances, left_hand_instances, right_hand_instances = estimate_pose(
                frame, person_boxes, pose_estimator_dwpose, visualizer_dwpose, show=False
            )
        else:
            raise ValueError(f"Unknown method: {method}. Use 'sapiens', 'rtmpose', or 'dwpose'.")

        # Visualize poses if needed
        if show or video_writer or frames_dir:
            vis_frame = visualize_wholebody_pose(
                frame, body_instances, left_hand_instances, right_hand_instances
            )
            
            if show:
                cv2.imshow('Wholebody Pose', vis_frame)
                cv2.resizeWindow('Wholebody Pose', 1980, 1080)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
            
            if video_writer:
                video_writer.write(vis_frame)
            elif frames_dir:
                frame_filename = os.path.join(frames_dir, f"frame_{frame_idx:06d}.png")
                cv2.imwrite(frame_filename, vis_frame)
        
        # Convert instances to serializable format and handle numpy types
        body_data = split_instances(body_instances) if body_instances is not None else []
        left_hand_data = split_instances(left_hand_instances) if left_hand_instances is not None else []
        right_hand_data = split_instances(right_hand_instances) if right_hand_instances is not None else []
        
        # Convert numpy values to native Python types
        body_instances_list.append(dict(
            frame_id=frame_idx,
            instances=convert_numpy_to_list(body_data)
        ))
        
        left_hand_instances_list.append(dict(
            frame_id=frame_idx,
            instances=convert_numpy_to_list(left_hand_data)
        ))
        
        right_hand_instances_list.append(dict(
            frame_id=frame_idx,
            instances=convert_numpy_to_list(right_hand_data)
        ))
        
    
    # Release resources
    cap.release()
    if video_writer:
        video_writer.release()
    if show:
        cv2.destroyAllWindows()
    
    # Save results to JSON file
    if pose_estimator is not None:  # Ensure model is initialized
        
        # Convert meta_info and all numpy arrays to lists
        meta_info = convert_numpy_to_list(pose_estimator.dataset_meta)

        results = {
            'meta_info': meta_info,
            'body_instances': body_instances_list,
            'left_hand_instances': left_hand_instances_list,
            'right_hand_instances': right_hand_instances_list
        }
        
        with open(output_json_path, 'w') as f:
            json.dump(results, f, indent=2)
            f.close()
        print(f"Predictions saved to {output_json_path}")
        if frames_dir:
            print(f"Visualization frames saved to {frames_dir}")
        elif video_writer:
            print(f"Visualization video saved to {output_path}")
        return results
    else:
        print("SAPIENS model not initialized. No results saved.")
        return None

## Process All Videos in Data Collection

Iterate through all video files in the data collection directory.

In [502]:
# Get list of video files in data directory
video_files = [f for f in os.listdir(data_dir) if f.endswith('.MP4') or f.endswith('.mp4')]

if not video_files:
    print(f"No video files found in {data_dir}")
else:
    print(f"Found {len(video_files)} video files")
    
    for video_file in video_files:
        video_name = os.path.splitext(video_file)[0]
        if video_name in [f'gopro{i}_synced_cut' for i in range(1, 5)]:
            print(f"Skipping far field camera: {video_name}")
            continue
        video_path = os.path.join(data_dir, video_file)
        for method in methods:
            

            output_json_path = os.path.join(pose_output_dir, method, data_collection, f"{video_name}_wholebody.json")
            output_frames_path = os.path.join(vis_output_dir, method, data_collection, f"{video_name}")

            # Skip if already processed
            if os.path.exists(output_json_path):
                print(f"Skipping {video_name} with {method} - already processed")
                continue
                
            print(f"Processing {video_name} with {method}...")
            try:
                process_video(video_path, output_json_path, output_path=None, stride=30, show=False, method=method)
                print(f"Completed processing {video_name}")
            except Exception as e:
                print(f"Error processing {video_name}: {str(e)}")
                import traceback
                traceback.print_exc()

Found 12 video files
Skipping gopro10_synced_cut with rtmpose - already processed
Skipping gopro10_synced_cut with dwpose - already processed
Skipping gopro11_synced_cut with rtmpose - already processed
Skipping gopro11_synced_cut with dwpose - already processed
Skipping gopro12_synced_cut with rtmpose - already processed
Skipping gopro12_synced_cut with dwpose - already processed
Skipping far field camera: gopro1_synced_cut
Skipping far field camera: gopro2_synced_cut
Skipping far field camera: gopro3_synced_cut
Skipping far field camera: gopro4_synced_cut
Skipping gopro5_synced_cut with rtmpose - already processed
Skipping gopro5_synced_cut with dwpose - already processed
Skipping gopro6_synced_cut with rtmpose - already processed
Skipping gopro6_synced_cut with dwpose - already processed
Skipping gopro7_synced_cut with rtmpose - already processed
Skipping gopro7_synced_cut with dwpose - already processed
Skipping gopro8_synced_cut with rtmpose - already processed
Skipping gopro8_syn

## Import refined poses from the hand pose files

In [503]:
def convert_hand_poses_to_json(hand_poses_dict, output_json_path, cam_names):
    """Convert hand poses dictionary to JSON format matching the structure above.
    
    Args:
        hand_poses_dict: Dictionary with structure {frame_idx: {cam_name: {obj_id: {'keypoints':np.array, 'scores': np.array}}}}
        output_json_path: Path to save the JSON file
        cam_names: List of camera names to include in the output
    """
    
    # Helper function to convert numpy arrays to lists recursively
    def convert_numpy_to_list(obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        elif isinstance(obj, dict):
            return {k: convert_numpy_to_list(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [convert_numpy_to_list(item) for item in obj]
        elif np.isscalar(obj) and isinstance(obj, (np.generic)):
            return obj.item()
        else:
            return obj
    
    # Initialize the result structure
    left_hand_instances = {}
    right_hand_instances = {}

    # Process each frame
    for frame_idx in sorted(hand_poses_dict.keys()):
        frame_data = hand_poses_dict[frame_idx]
        
        # Process each camera
        for cam_name in cam_names:
            if cam_name in frame_data:
                cam_data = frame_data[cam_name]
            else:
                cam_data = {0: {}, 1: {}}
            frame_instances_left = []
            frame_instances_right = []
            # Process each object/person
            for obj_id, pose_data in cam_data.items():
                # Extract keypoints and scores
                keypoints = pose_data.get('keypoints', np.array([]))
                scores = pose_data.get('scores', np.array([]))
                
                if len(keypoints) == 0 or len(scores) == 0:
                    continue
                # Create instance data structure
                instance = {
                    'bbox': [],  # Empty bbox as not provided in input
                    'bbox_score': 1.0,
                    'keypoints': keypoints.squeeze(),
                    'keypoint_scores': scores.squeeze(),
                }
                
                if obj_id == 0:
                    frame_instances_left.append(convert_numpy_to_list(instance))
                else:
                    frame_instances_right.append(convert_numpy_to_list(instance))

            if cam_name not in left_hand_instances and cam_name not in right_hand_instances:
                left_hand_instances[cam_name] = []
                right_hand_instances[cam_name] = []

            # Add frame data to the list
            left_hand_instances[cam_name].append({
                'frame_id': int(frame_idx),
                'instances': frame_instances_left
            })
            right_hand_instances[cam_name].append({
                'frame_id': int(frame_idx),
                'instances': frame_instances_right
            })

    # Create meta_info (basic structure for hand poses)
    meta_info = {
        'dataset_name': 'hand_poses',
        'paper_info': {},
        'keypoint_info': {
            'skeleton_info': [],
            'joint_weights': [1.0] * 21,  # Standard 21 hand keypoints
            'sigmas': []
        }
    }

    for cam_name in left_hand_instances.keys():
        cam_left_instances = left_hand_instances[cam_name]
        cam_right_instances = right_hand_instances[cam_name]
        # Create the final result structure
        results = {
            'meta_info': meta_info,
            'left_hand_instances': cam_left_instances,
            'right_hand_instances': cam_right_instances,
            'body_instances': [{'instances': []} for instance in cam_left_instances]
        }

        # Save to JSON file
        with open(os.path.join(output_frames_path, f'{cam_name}_synced_cut_wholebody.json'), 'w+') as f:
            json.dump(results, f, indent=2)

    print(f"Hand poses saved to {output_json_path}")
    return results

# Example usage:
# Assuming you have your hand_poses_dict ready
def load_hand_poses(data_collection, labeled_frames):
    # Path to labeled 2D hand poses (ground truth)
    labeled_2d_path = f'output_3d/{data_collection}/hand_poses_2d.npz'  # Update as needed
    # Load 2D Labeled Hand Poses
    data_2d = np.load(labeled_2d_path, allow_pickle=True)
    # Assume the file contains a dict: {cam_name: {frame_idx: {obj_id: {'keypoints': ndarray, 'scores': ndarray}}}}
    unindexed_poses_2d = data_2d['poses_2d'].item() if 'poses_2d' in data_2d else data_2d[list(data_2d.keys())[0]].item()

    cam_names = list(unindexed_poses_2d.keys())
    poses_2d = {k: {cam_name: {} for cam_name in cam_names} for k in labeled_frames}
    for cam_name in cam_names:
        for frame_idx in labeled_frames:
            for obj_id in unindexed_poses_2d[cam_name][frame_idx].keys():
                keypoints = unindexed_poses_2d[cam_name][frame_idx][obj_id]['keypoints']
                scores = unindexed_poses_2d[cam_name][frame_idx][obj_id]['keypoint_scores']
                # Ensure keypoints and scores are numpy arrays
                poses_2d[frame_idx][cam_name][obj_id] = {'keypoints':np.array(keypoints), 'scores': np.array(scores)}
    # print(f"Loaded labeled 2D hand poses for {len(labeled_2d_poses)} frames.")

    return poses_2d

poses_2d = load_hand_poses(data_collection, labeled_frames)
os.makedirs(os.path.join('predictions_2d', 'refined_poses', data_collection), exist_ok=True)
output_frames_path = os.path.join('predictions_2d', 'refined_poses', data_collection)
convert_hand_poses_to_json(poses_2d, output_frames_path, [f"gopro{idx}" for idx in range(5, 13)])


Hand poses saved to predictions_2d\refined_poses\cha/synced_cut


{'meta_info': {'dataset_name': 'hand_poses',
  'paper_info': {},
  'keypoint_info': {'skeleton_info': [],
   'joint_weights': [1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0,
    1.0],
   'sigmas': []}},
 'left_hand_instances': [{'frame_id': 0,
   'instances': [{'bbox': [],
     'bbox_score': 1.0,
     'keypoints': [[1757.80078125, 875.3125],
      [1719.177734375, 885.908203125],
      [1683.630859375, 899.921875],
      [1669.275390625, 930.68359375],
      [1654.236328125, 952.216796875],
      [1724.646484375, 955.634765625],
      [1690.466796875, 980.5859375],
      [1665.173828125, 980.927734375],
      [1647.7421875, 976.142578125],
      [1740.7109375, 966.9140625],
      [1700.37890625, 992.20703125],
      [1675.0859375, 984.00390625],
      [1656.62890625, 974.775390625],
      [1745.49609375, 967.255859375],
      [1709.265625, 984.00390625],
    